# Costruzione decrizione automatica svg automa

## Esempio estrazione

In [49]:
import xml.etree.ElementTree as ET
import re

def apri_file_svg(nome_file):
    tree = ET.parse(nome_file)
    root = tree.getroot()
    return root

# Esempio di utilizzo
nome_file_svg = 'automa.svg'
root_svg = apri_file_svg(nome_file_svg)

for child in root_svg:
    if child.attrib['id'] == 'layer1':
        automa1 = child

#foreach element in automa1 print id
for child in automa1:
    if not re.search('title', child.attrib['id']):
        print('RAMO: ', child.attrib['id'])

        #forreach element in child print id
        for child2 in child:
            if not re.search('title', child2.attrib['id']):
                print('- ', child2.attrib['id'])
                
                #get text by id valore-r6
                for child3 in child2:
                    if re.search('valore', child3.attrib['id']):
                        for child4 in child3:
                            print('---- ', child4.text)


    print('===============================')

RAMO:  stato-q4
-  simbolo-q4
-  nome-q4
RAMO:  stato-q3
-  simbolo-q3
-  nome-q3
RAMO:  stato-q2
-  simbolo-q2
-  nome-q2
RAMO:  stato-q1
-  simbolo-q1
-  nome-q1
RAMO:  stato-q0-finale
-  simbolo-q0
-  simbolo-q0-0
-  nome-q0
RAMO:  transizione-q4-q0
-  simbolo-q4-q0
-  valore-q4-q0
RAMO:  transizione-q3-q4
-  simbolo-q3-q4
-  valore-q3-q4
RAMO:  transizione-q2-q3
-  simbolo-q2-q3
-  valore-q2-q3
RAMO:  transizione-q1-q2
-  simbolo-q1-q2
-  valore-q1-q2
RAMO:  transizione-q0-q1
-  simbolo-q0-q1
-  valore-q0-q1
RAMO:  start-q0
-  simbolo-start
-  nome-start


## GENERAZIONE REGOLE

## ESTRAZIONE DEL DIZIONARIO

In [50]:
import xml.etree.ElementTree as ET
import re

def apri_file_svg(nome_file):
    tree = ET.parse(nome_file)
    root = tree.getroot()
    return root

# Esempio di utilizzo
nome_file_svg = 'automa.svg'
root_svg = apri_file_svg(nome_file_svg)

# Trova l'elemento con id="automa1"
for child in root_svg:
    if child.attrib['id'] == 'layer1':
        automa1 = child


stati = []
linguaggio = []
transizioni = []
transizioni_linguaggio = []
stato_iniziale = ''
stato_finale = ''

for child in automa1:
    if not re.search('title', child.attrib['id']):
        elemento = child.attrib['id']

        if re.search('start', elemento):
            elemento = elemento.replace('start-', '')
            stato_iniziale = elemento

        if re.search('stato', elemento):
            elemento = elemento.replace('stato-', '')

            if re.search('finale', elemento):
                elemento = elemento.replace('-finale', '')
                stato_finale = elemento

            stati.append(elemento)

        if re.search('transizione', elemento):
            elemento = elemento.replace('transizione-', '')
            elemento = elemento.split('-')
            transizioni.append(elemento)

            for child2 in child:
                if re.search('valore', child2.attrib['id']):
                    for child3 in child2:
                        linguaggio.append(child3.text)

                        transizioni_linguaggio.append([elemento, child3.text])

# revert stati
stati = stati[::-1]
# revert linguaggio
linguaggio = linguaggio[::-1]
# revert transizioni
transizioni = transizioni[::-1]
# revert transizioni_linguaggio
transizioni_linguaggio = transizioni_linguaggio[::-1]


print('stato_iniziale: ', stato_iniziale)
print('stato_finale: ', stato_finale)
print('stati: ', stati)
print('linguaggio: ', linguaggio)
print('transizioni: ', transizioni)
print('transizioni_linguaggio: ', transizioni_linguaggio)

stato_iniziale:  q0
stato_finale:  q0
stati:  ['q0', 'q1', 'q2', 'q3', 'q4']
linguaggio:  ['1', '1', '0', '0', '0']
transizioni:  [['q0', 'q1'], ['q1', 'q2'], ['q2', 'q3'], ['q3', 'q4'], ['q4', 'q0']]
transizioni_linguaggio:  [[['q0', 'q1'], '1'], [['q1', 'q2'], '1'], [['q2', 'q3'], '0'], [['q3', 'q4'], '0'], [['q4', 'q0'], '0']]


In [51]:
svg = '<image>automi/automa.svg</image>'
intent = 'fsa-practical'
risposte = {}

In [52]:
categories = []

### In generale

```xml
<category intent="fsa-practical" argument="automaton">
    <acts>
        <act time="0">Ta:request</act>
    </acts>
    <frame></frame>
    <template>
        L'automa accetta zero o più parole formate da una sequenza del tipo 11000. In totale ci sono 5 stati: q0, q1, q2, q3 e q4. q0 è 
        sia lo stato iniziale che lo stato finale.Le transizioni sono: q0 con valore 1 va in q1, q1 con valore 1 va in q2, q2 con valore
        0 va in q3, q3 con valore 0 va in q4, q4 con valore 0 va in q0.
        <image>automi/automa.svg</image>
    </template>
</category>

In [53]:
category = {
    "intent": "fsa-practical",
    "argument": "automaton",
    "acts": {
        0: "Ta:request",
    },
    "frame": "<frame></frame>",
    "template": ""
}

text = 'The automaton accepts zero or more words formed by a sequence of the type '

# linguaggio
text += ''.join(linguaggio) + '. '

# stati
if (len(stati) > 1):
    text += '<MUTATION>In total there are ' + str(len(stati)) + ' states: '
else:
    text += '<MUTATION>In total there is only one state: '
for i in range(len(stati)):
    if (i == len(stati) - 1):
        text += stati[i] + '. '
    elif (i == len(stati) - 2):
        text += stati[i] + ' e '
    #else:
    #    text += stati[i] + ', '

# stato iniziale e stato finale
if (stato_iniziale != '' and stato_finale != ''):
    if (stato_iniziale == stato_finale):
        text += stato_iniziale + ' is both the initial and the final state.'
    else:
        text += 'The initial state is ' + stato_iniziale + ' and the final state is ' + stato_finale + '.'

# transizioni
text += 'Le transizioni sono: '
for i in range(len(transizioni)):
    if (i == len(transizioni) - 1):
        text += transizioni[i][0] + ' with value ' + transizioni_linguaggio[i][1] + ' goes to ' + transizioni[i][1] + '.'
    else:
        text += transizioni[i][0] + ' with value ' + transizioni_linguaggio[i][1] + ' goes goes ' + transizioni[i][1] + ', '

text += svg

category['template'] = text
categories.append(category)

category

{'intent': 'fsa-practical',
 'argument': 'automaton',
 'acts': {0: 'Ta:request'},
 'frame': '<frame></frame>',
 'template': 'The automaton accepts zero or more words formed by a sequence of the type 11000. <MUTATION>In total there are 5 states: q3 e q4. q0 is both the initial and the final state.Le transizioni sono: q0 with value 1 goes goes q1, q1 with value 1 goes goes q2, q2 with value 0 goes goes q3, q3 with value 0 goes goes q4, q4 with value 0 goes to q0.<image>automi/automa.svg</image>'}

### Stati

In [54]:
# Quali sono gli stati
category = {
    "intent": "fsa-practical",
    "argument": "state",
    "acts": {
        0: "Ta:request",
    },
    "frame": "<frame></frame>",
    "template": ""
}

text = 'The automaton has ' + str(len(stati)) + ' states: '
for i in range(len(stati)):
    if (i == len(stati) - 1):
        text += stati[i] + '.'
    elif (i == len(stati) - 2):
        text += stati[i] + ' e '
    else:
        text += stati[i] + ', '

for stato in stati:
    elemento = 'simbolo-' + stato
    text += '<svgElement style-name="stroke" style-value="#04ed00">' + elemento + '</svgElement>'

text += svg


category['template'] = text
categories.append(category)

category

{'intent': 'fsa-practical',
 'argument': 'state',
 'acts': {0: 'Ta:request'},
 'frame': '<frame></frame>',
 'template': 'The automaton has 5 states: q0, q1, q2, q3 e q4.<svgElement style-name="stroke" style-value="#04ed00">simbolo-q0</svgElement><svgElement style-name="stroke" style-value="#04ed00">simbolo-q1</svgElement><svgElement style-name="stroke" style-value="#04ed00">simbolo-q2</svgElement><svgElement style-name="stroke" style-value="#04ed00">simbolo-q3</svgElement><svgElement style-name="stroke" style-value="#04ed00">simbolo-q4</svgElement><image>automi/automa.svg</image>'}

### Transizioni

In [55]:
# quali sono le transizioni
category = {
    "intent": "fsa-practical",
    "argument": "transition",
    "acts": {
        0: "Ta:request",
    },
    "frame": """<frame></frame>""",
    "template": ""
}

text = 'The automaton has ' + str(len(transizioni)) + ' transitions: '
for i in range(len(transizioni)):
    if (i == len(transizioni) - 1):
        text += transizioni[i][0] + ' con valore ' + transizioni_linguaggio[i][1] + ' va in ' + transizioni[i][1] + '.'
    else:
        text += transizioni[i][0] + ' con valore ' + transizioni_linguaggio[i][1] + ' va in ' + transizioni[i][1] + ', '


for transizione in transizioni:
    elemento = 'simbolo-' + transizione[0] + '-' + transizione[1]
    text += '<svgElement style-name="stroke" style-value="#04ed00">' + elemento + '</svgElement>'

text += svg


category['template'] = text
categories.append(category)

category

{'intent': 'fsa-practical',
 'argument': 'transition',
 'acts': {0: 'Ta:request'},
 'frame': '<frame></frame>',
 'template': 'The automaton has 5 transitions: q0 con valore 1 va in q1, q1 con valore 1 va in q2, q2 con valore 0 va in q3, q3 con valore 0 va in q4, q4 con valore 0 va in q0.<svgElement style-name="stroke" style-value="#04ed00">simbolo-q0-q1</svgElement><svgElement style-name="stroke" style-value="#04ed00">simbolo-q1-q2</svgElement><svgElement style-name="stroke" style-value="#04ed00">simbolo-q2-q3</svgElement><svgElement style-name="stroke" style-value="#04ed00">simbolo-q3-q4</svgElement><svgElement style-name="stroke" style-value="#04ed00">simbolo-q4-q0</svgElement><image>automi/automa.svg</image>'}

### Stato iniziale

In [56]:
# qual è lo stato iniziale
category = {
    "intent": "fsa-practical",
    "argument": "state",
    "acts": {
        0: "Ta:request",
    },
    "frame": f"""
        <frame>
            <slot name="initialState" value="?"/>
        </frame>
    """,
    "template": ""
}


text = 'The initial state is ' + stato_iniziale + '.'
elemento = 'simbolo-' + stato_iniziale
text += '<svgElement style-name="stroke" style-value="#04ed00">' + elemento + '</svgElement>'
text += svg

category['template'] = text
categories.append(category)

category

{'intent': 'fsa-practical',
 'argument': 'state',
 'acts': {0: 'Ta:request'},
 'frame': '\n        <frame>\n            <slot name="initialState" value="?"/>\n        </frame>\n    ',
 'template': 'The initial state is q0.<svgElement style-name="stroke" style-value="#04ed00">simbolo-q0</svgElement><image>automi/automa.svg</image>'}

### Stato finale

In [57]:
# qual 'è lo stato finale?
category = {
    "intent": "fsa-practical",
    "argument": "state",
    "acts": {
        0: "Ta:request",
    },
    "frame": f"""
        <frame>
            <slot name="finalStates" value="?"/>
        </frame>
    """,
    "template": ""
}

text = 'The final state is ' + stato_finale + '.'
elemento = 'simbolo-' + stato_finale
text += '<svgElement style-name="stroke" style-value="#04ed00">' + elemento + '</svgElement>'
text += svg

category['template'] = text
categories.append(category)

category

{'intent': 'fsa-practical',
 'argument': 'state',
 'acts': {0: 'Ta:request'},
 'frame': '\n        <frame>\n            <slot name="finalStates" value="?"/>\n        </frame>\n    ',
 'template': 'The final state is q0.<svgElement style-name="stroke" style-value="#04ed00">simbolo-q0</svgElement><image>automi/automa.svg</image>'}

### Stati e archi

In [58]:
# quanti stati iniziali e finali ci sono
category = {
    "intent": "fsa-practical",
    "argument": "automaton",
    "acts": {
        0: "Ta:request",
    },
    "frame": f"""
        <frame>
            <slot name="numberOfStates" value="?"/>
            <slot name="numberOfTransitions" value="?"/>
        </frame>
    """,
    "template": ""
}

text = 'The automaton has ' + str(len(stati)) + ' states and ' + str(len(transizioni)) + ' transitions.'
text += svg

category['template'] = text
categories.append(category)

category

{'intent': 'fsa-practical',
 'argument': 'automaton',
 'acts': {0: 'Ta:request'},
 'frame': '\n        <frame>\n            <slot name="numberOfStates" value="?"/>\n            <slot name="numberOfTransitions" value="?"/>\n        </frame>\n    ',
 'template': 'The automaton has 5 states and 5 transitions.<image>automi/automa.svg</image>'}

### Stato iniziale e stato finale ###

In [59]:
# qual è lo stato finale e quale lo stato iniziale
category = {
    "intent": "fsa-practical",
    "argument": "state",
    "acts": {
        0: "Ta:request",
    },
    "frame": f"""
        <frame>
            <slot name="initialState" value="?""/>
            <slot name="finalStates" value="?"/>
        </frame>
    """,
    "template": ""
}

text = ''

# stato iniziale e stato finale
if (stato_iniziale != '' and stato_finale != ''):
    if (stato_iniziale == stato_finale):
        text += stato_iniziale + ' is the initial and the final state.'
    else:
        text += 'The initial state is ' + stato_iniziale + ' and the final state is ' + stato_finale + '.'
    elemento = 'simbolo-' + stato_iniziale
    text += '<svgElement style-name="stroke" style-value="#04ed00">' + elemento + '</svgElement>'

    if (not (stato_iniziale == stato_finale)):
        elemento2 = 'simbolo-' + stato_finale
        text += '<svgElement style-name="stroke" style-value="#04ed00">' + elemento2 + '</svgElement>'
else:
    text += "the initial state and the final state are not defined. "
    
text += svg

category['template'] = text
categories.append(category)

category

{'intent': 'fsa-practical',
 'argument': 'state',
 'acts': {0: 'Ta:request'},
 'frame': '\n        <frame>\n            <slot name="initialState" value="?""/>\n            <slot name="finalStates" value="?"/>\n        </frame>\n    ',
 'template': 'q0 is the initial and the final state.<svgElement style-name="stroke" style-value="#04ed00">simbolo-q0</svgElement><image>automi/automa.svg</image>'}

### Da ... a ...

In [60]:
for stato in stati:
    for stato2 in stati:
        category = {
            "intent": "fsa-practical",
            "argument": "transition",
            "acts": {
                0: "Ta:propositionalQuestion",
            },
            "frame": f"""
                <frame>
                    <slot name="transitions">
                        <slot-values>
                            <slot-value value="{stato}"/>
                            <slot-value value="{stato2}"/>
                            <slot-value value="?"/>
                        </slot-values>
                    </slot>
                </frame>
            """,
            "template": ""
        }

        flag = False
        valore = ''
        for s, value in transizioni_linguaggio:
            if (s == [stato, stato2]):
                flag = True
                valore = value

        if (flag):
            text = 'The transition between ' + stato + ' and ' + stato2 + ' is with value ' + valore + '.'
            elemento = 'simbolo-' + stato + '-' + stato2
            text += '<svgElement style-name="stroke" style-value="#04ed00">' + elemento + '</svgElement>'
        else:
            text = 'there is no transition between ' + stato + ' and ' + stato2 + '.'

        text += svg

        category['template'] = text
        categories.append(category)

category

{'intent': 'fsa-practical',
 'argument': 'transition',
 'acts': {0: 'Ta:propositionalQuestion'},
 'frame': '\n                <frame>\n                    <slot name="transitions">\n                        <slot-values>\n                            <slot-value value="q4"/>\n                            <slot-value value="q4"/>\n                            <slot-value value="?"/>\n                        </slot-values>\n                    </slot>\n                </frame>\n            ',
 'template': 'there is no transition between q4 and q4.<image>automi/automa.svg</image>'}

### No transazione in quanto non esistono entrambi gli stati

In [61]:
# transizione tra stato X e stato Y
category = {
    "intent": "fsa-practical",
    "argument": "transition",
    "acts": {
        0: "Ta:request",
    },
    "frame": f"""
        <frame>
            <slot name="transitions">
                <slot-values>
                    <slot-value value="*"/>
                    <slot-value value="*"/>
                    <slot-value value="?"/>
                </slot-values>
            </slot>
        </frame>
    """,
    "template": ""
}

text = 'the transition is not defined.'
text += svg

category['template'] = text
categories.append(category)

category

{'intent': 'fsa-practical',
 'argument': 'transition',
 'acts': {0: 'Ta:request'},
 'frame': '\n        <frame>\n            <slot name="transitions">\n                <slot-values>\n                    <slot-value value="*"/>\n                    <slot-value value="*"/>\n                    <slot-value value="?"/>\n                </slot-values>\n            </slot>\n        </frame>\n    ',
 'template': 'the transition is not defined.<image>automi/automa.svg</image>'}

### No transazione sicura in quanto non esiste lo stato di arrivo

In [62]:
for stato in stati:
    # transizione tra stato X e stato Y
    category = {
        "intent": "fsa-practical",
        "argument": "transition",
        "acts": {
            0: "Ta:request",
        },
        "frame": f"""
            <frame>
                <slot name="transitions">
                    <slot-values>
                        <slot-value value="{stato}"/>
                        <slot-value value="*"/>
                        <slot-value value="?"/>
                    </slot-values>
                </slot>
            </frame>
        """,
        "template": ""
    }

    text = 'the transition is not defined.'
    text += svg

    category['template'] = text
    categories.append(category)

category

{'intent': 'fsa-practical',
 'argument': 'transition',
 'acts': {0: 'Ta:request'},
 'frame': '\n            <frame>\n                <slot name="transitions">\n                    <slot-values>\n                        <slot-value value="q4"/>\n                        <slot-value value="*"/>\n                        <slot-value value="?"/>\n                    </slot-values>\n                </slot>\n            </frame>\n        ',
 'template': 'the transition is not defined.<image>automi/automa.svg</image>'}

### No transazione sicura in quanto non esiste lo stato di partenza

In [63]:
for stato in stati:
    # transizione tra stato X e stato Y
    category = {
        "intent": "fsa-practical",
        "argument": "transition",
        "acts": {
            0: "Ta:request",
        },
        "frame": f"""
            <frame>
                <slot name="transitions">
                    <slot-values>
                        <slot-value value="*"/>
                        <slot-value value="{stato}"/>
                        <slot-value value="?"/>
                    </slot-values>
                </slot>
            </frame>
        """,
        "template": ""
    }

    text = 'the transition is not defined.'
    text += svg

    category['template'] = text
    categories.append(category)

category

{'intent': 'fsa-practical',
 'argument': 'transition',
 'acts': {0: 'Ta:request'},
 'frame': '\n            <frame>\n                <slot name="transitions">\n                    <slot-values>\n                        <slot-value value="*"/>\n                        <slot-value value="q4"/>\n                        <slot-value value="?"/>\n                    </slot-values>\n                </slot>\n            </frame>\n        ',
 'template': 'the transition is not defined.<image>automi/automa.svg</image>'}

### ESISTE...

#### Stati esistenti

In [64]:
for stato in stati:
    category = {
        "intent": "fsa-practical",
        "argument": "state",
        "acts": {
            0: "Ta:propositionalQuestion",
        },
        "frame": f"""
            <frame>
                <slot name="states">
                    <slot-value value="{stato}"/>
                </slot>
            </frame>
        """,
        "template": ""
    }

    text = f'The state {stato} exists.' 
    elemento = 'simbolo-' + stato
    text += '<svgElement style-name="stroke" style-value="#04ed00">' + elemento + '</svgElement>'
    text += svg

    category['template'] = text
    categories.append(category)

category

{'intent': 'fsa-practical',
 'argument': 'state',
 'acts': {0: 'Ta:propositionalQuestion'},
 'frame': '\n            <frame>\n                <slot name="states">\n                    <slot-value value="q4"/>\n                </slot>\n            </frame>\n        ',
 'template': 'The state q4 exists.<svgElement style-name="stroke" style-value="#04ed00">simbolo-q4</svgElement><image>automi/automa.svg</image>'}

#### Stato iniziale esiste

In [65]:
category = {
    "intent": "fsa-practical",
    "argument": "state",
    "acts": {
        0: "Ta:propositionalQuestion",
    },
    "frame": f"""
        <frame>
            <slot name="initialState" value="{stato_iniziale}"/>
        </frame>
    """,
    "template": ""
}

text = 'The initial state is ' + stato_iniziale + '.'
elemento = 'simbolo-' + stato_iniziale
text += '<svgElement style-name="stroke" style-value="#04ed00">' + elemento + '</svgElement>'
text += svg

category['template'] = text
categories.append(category)

category

{'intent': 'fsa-practical',
 'argument': 'state',
 'acts': {0: 'Ta:propositionalQuestion'},
 'frame': '\n        <frame>\n            <slot name="initialState" value="q0"/>\n        </frame>\n    ',
 'template': 'The initial state is q0.<svgElement style-name="stroke" style-value="#04ed00">simbolo-q0</svgElement><image>automi/automa.svg</image>'}

#### Stato iniziale non esiste

In [66]:
category = {
    "intent": "fsa-practical",
    "argument": "state",
    "acts": {
        0: "Ta:propositionalQuestion",
    },
    "frame": f"""
        <frame>
            <slot name="initialState" value="*"/>
        </frame>
    """,
    "template": ""
}

text = 'It is not the initial state.'
text += svg

category['template'] = text
categories.append(category)

category

{'intent': 'fsa-practical',
 'argument': 'state',
 'acts': {0: 'Ta:propositionalQuestion'},
 'frame': '\n        <frame>\n            <slot name="initialState" value="*"/>\n        </frame>\n    ',
 'template': 'It is not the initial state.<image>automi/automa.svg</image>'}

#### Stato finale

In [67]:
category = {
    "intent": "fsa-practical",
    "argument": "state",
    "acts": {
        0: "Ta:propositionalQuestion",
    },
    "frame": f"""
        <frame>
            <slot name="finalStates">
                <slot-value value="{stato_finale}"/>
            </slot>
        </frame>
    """,
    "template": ""
}



text = 'The final state is ' + stato_finale + '.'
elemento = 'simbolo-' + stato_finale
text += '<svgElement style-name="stroke" style-value="#04ed00">' + elemento + '</svgElement>'
text += svg

category['template'] = text
categories.append(category)
category

{'intent': 'fsa-practical',
 'argument': 'state',
 'acts': {0: 'Ta:propositionalQuestion'},
 'frame': '\n        <frame>\n            <slot name="finalStates">\n                <slot-value value="q0"/>\n            </slot>\n        </frame>\n    ',
 'template': 'The final state is q0.<svgElement style-name="stroke" style-value="#04ed00">simbolo-q0</svgElement><image>automi/automa.svg</image>'}

#### Non esistente

In [68]:
category = {
    "intent": "fsa-practical",
    "argument": "state",
    "acts": {
        0: "Ta:propositionalQuestion",
    },
    "frame": f"""
        <frame>
            <slot name="finalStates">
                <slot-value value="*"/>
            </slot>
        </frame>
    """,
    "template": ""
}



text = 'The state is not the final state.'
text += svg

category['template'] = text
categories.append(category)
category

{'intent': 'fsa-practical',
 'argument': 'state',
 'acts': {0: 'Ta:propositionalQuestion'},
 'frame': '\n        <frame>\n            <slot name="finalStates">\n                <slot-value value="*"/>\n            </slot>\n        </frame>\n    ',
 'template': 'The state is not the final state.<image>automi/automa.svg</image>'}

### Quanti stati

In [69]:
category = {
    "intent": "fsa-practical",
    "argument": "state",
    "acts": {
        0: "Ta:request",
    },
    "frame": f"""
        <frame>
            <slot name="numberOfStates" value="?"/>
        </frame>
    """,
    "template": ""
}

text = 'This automaton has ' + str(len(stati)) + ' states'

for stato in stati:
    elemento = 'simbolo-' + stato
    text += '<svgElement style-name="stroke" style-value="#04ed00">' + elemento + '</svgElement>'
text += svg

category['template'] = text
categories.append(category)

category

{'intent': 'fsa-practical',
 'argument': 'state',
 'acts': {0: 'Ta:request'},
 'frame': '\n        <frame>\n            <slot name="numberOfStates" value="?"/>\n        </frame>\n    ',
 'template': 'This automaton has 5 states<svgElement style-name="stroke" style-value="#04ed00">simbolo-q0</svgElement><svgElement style-name="stroke" style-value="#04ed00">simbolo-q1</svgElement><svgElement style-name="stroke" style-value="#04ed00">simbolo-q2</svgElement><svgElement style-name="stroke" style-value="#04ed00">simbolo-q3</svgElement><svgElement style-name="stroke" style-value="#04ed00">simbolo-q4</svgElement><image>automi/automa.svg</image>'}

### Gli stati sono ... (risposta giusta)

In [70]:
category = {
    "intent": "fsa-practical",
    "argument": "state",
    "acts": {
        0: "Ta:propositionalQuestion",
    },
    "frame": f"""
        <frame>
            <slot name="numberOfStates" value="{str(len(stati))}"/>
        </frame>
    """,
    "template": ""
}

text = f'There are {str(len(stati))} states in this automaton.'

for stato in stati:
    elemento = 'simbolo-' + stato
    text += '<svgElement style-name="stroke" style-value="#04ed00">' + elemento + '</svgElement>'
text += svg

category['template'] = text
categories.append(category)

category

{'intent': 'fsa-practical',
 'argument': 'state',
 'acts': {0: 'Ta:propositionalQuestion'},
 'frame': '\n        <frame>\n            <slot name="numberOfStates" value="5"/>\n        </frame>\n    ',
 'template': 'There are 5 states in this automaton.<svgElement style-name="stroke" style-value="#04ed00">simbolo-q0</svgElement><svgElement style-name="stroke" style-value="#04ed00">simbolo-q1</svgElement><svgElement style-name="stroke" style-value="#04ed00">simbolo-q2</svgElement><svgElement style-name="stroke" style-value="#04ed00">simbolo-q3</svgElement><svgElement style-name="stroke" style-value="#04ed00">simbolo-q4</svgElement><image>automi/automa.svg</image>'}

### Gli stati sono ... (risposta sbagliata)

In [71]:
category = {
    "intent": "fsa-practical",
    "argument": "state",
    "acts": {
        0: "Ta:propositionalQuestion",
    },
    "frame": f"""
        <frame>
            <slot name="numberOfStates" value="*"/>
        </frame>
    """,
    "template": ""
}

text = 'The number of states is incorrect.'

for stato in stati:
    elemento = 'simbolo-' + stato
    text += '<svgElement style-name="stroke" style-value="#04ed00">' + elemento + '</svgElement>'
text += svg

category['template'] = text
categories.append(category)

category

{'intent': 'fsa-practical',
 'argument': 'state',
 'acts': {0: 'Ta:propositionalQuestion'},
 'frame': '\n        <frame>\n            <slot name="numberOfStates" value="*"/>\n        </frame>\n    ',
 'template': 'The number of states is incorrect.<svgElement style-name="stroke" style-value="#04ed00">simbolo-q0</svgElement><svgElement style-name="stroke" style-value="#04ed00">simbolo-q1</svgElement><svgElement style-name="stroke" style-value="#04ed00">simbolo-q2</svgElement><svgElement style-name="stroke" style-value="#04ed00">simbolo-q3</svgElement><svgElement style-name="stroke" style-value="#04ed00">simbolo-q4</svgElement><image>automi/automa.svg</image>'}

## Generazione AIML

In [72]:
import xml.etree.ElementTree as ET
import xml.dom.minidom as minidom

def build_aiml(categories, out_path):
    root = ET.Element('aiml')

    for cat in categories:
        c = ET.SubElement(root, 'category')
        if 'intent' in cat:
            c.set('intent', cat['intent'])
        if 'argument' in cat:
            c.set('argument', cat['argument'])

        # acts
        acts_el = ET.SubElement(c, 'acts')
        for time, act in cat.get('acts', {}).items():
            act_el = ET.SubElement(acts_el, 'act', {'time': str(time)})
            act_el.text = act

        # frame: proviamo a parsare come XML, altrimenti lo inseriamo come testo dentro <frame>
        frame_str = cat.get('frame', '').strip()
        if frame_str:
            try:
                frame_elem = ET.fromstring(frame_str)
            except ET.ParseError:
                # fallback: crea frame e metti dentro il testo raw
                f_el = ET.SubElement(c, 'frame')
                f_el.text = frame_str
            else:
                # append dell'elemento frame già parsato (mantiene eventuali slot/attributi)
                c.append(frame_elem)
        else:
            # assicuriamoci che esista comunque <frame></frame>
            ET.SubElement(c, 'frame')

        # template: se il testo della template è un frammento XML valido, lo parsiamo e trasferiamo i figli
        template_el = ET.SubElement(c, 'template')
        tpl = cat.get('template', '')

        if tpl.strip():
            # wrapper per poter parsare testo misto + tag
            try:
                fragment = ET.fromstring(f'<fragment>{tpl}</fragment>')
            except ET.ParseError:
                # fallback: testo semplice (verrà escape-ato correttamente)
                template_el.text = tpl
            else:
                # testo prima del primo figlio
                template_el.text = fragment.text
                # append di ogni figlio (con le eventuali .tail preservate)
                for child in list(fragment):
                    template_el.append(child)
        # altrimenti template rimane vuoto

    # Serializziamo con minidom per ottenere pretty print
    rough = ET.tostring(root, encoding='utf-8')
    dom = minidom.parseString(rough)

    # Forza <frame></frame> (senza self-closing) aggiungendo un textnode vuoto quando il frame è veramente vuoto
    for frame_node in dom.getElementsByTagName('frame'):
        if not frame_node.hasChildNodes():
            frame_node.appendChild(dom.createTextNode(''))

    pretty = dom.toprettyxml(indent='    ', encoding='utf-8')
    with open(out_path, 'wb') as f:
        f.write(pretty)

# Esegui la generazione
file_path = 'automa.aiml'
build_aiml(categories, file_path)
print(f"✅ File AIML+ generato in: {file_path}")

✅ File AIML+ generato in: automa.aiml
